***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.1 作为最小二乘问题的校准](8_1_calibration_least_squares_problem.ipynb)
    * 下一节： [8.3 第二代校准（2GC）](8_3_2gc.ipynb)

***

导入标准模块:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

导入本节所需的专用模块:

In [2]:
%pylab inline
pylab.rcParams['figure.figsize'] = (18, 6)

Populating the interactive namespace from numpy and matplotlib


In [3]:
HTML('../style/code_toggle.html')

## 8.2 第一代校准（1GC）<a id='cal:sec:1gccal'></a>

前面的章节已经反复说明，干涉仪并不会给出无误差的测量结果。相反，观测数据会受到多种效应的污染，包括环境效应（例如大气）、系统故障以及仪器本身的不完美。做好科学分析的前提是得到尽可能可靠的数据，而校准的目的，正是尽可能把这些受损信号恢复到接近真实的状态。

我们讨论校准时的起点，就是 1GC，也就是第一代校准。它通常是整个校准流程中的第一步，而实现 1GC 的核心，就是进行校准源观测。

## 8.2.1 校准源观测<a id='cal:sec:calobs'></a>

所谓校准源观测，就是观测某些参数已知的源，例如已知其流量、形状或谱分布。通常会把这些校准源观测穿插在目标场观测之间，以便追踪观测参数随时间的变化。这样我们就能从校准源数据中求得增益解，并把这些解转移到目标场。这通常是去除可见度中大尺度误差的一种非常有效的方法。具体采用什么样的校准源，则取决于我们想校准的是什么量。

<p class=conclusion>
  <font size=4> <b>常见校准量</b></font>
  <br>
  <br>
&bull; **增益校准**：用于确定复增益，并且需要在整个观测过程中反复进行。校准源应当相对较亮，更重要的是要尽可能靠近目标场，因为增益校准往往用于追踪大气等本地效应的变化。校准源离目标越近，它与目标场经历相同环境效应的可能性就越大。<br><br>

&bull; **绝对流量校准**：用于确定视场中各源的真实流量刻度，保证得到的流量值在数量级上是正确的。这类校准通常要求非常亮、稳定、近似点状或能够被良好建模的校准源。<br><br>

&bull; **带通校准**：用于改正观测频率方向上的误差，这些误差既可能来自系统，也可能来自大气。带通校准需要一个非常亮、稳定、近似点状或可准确建模、并且具有已知谱形的源。满足这些条件的源数量很少，而且往往离目标场很远。<br><br>

&bull; **延迟校准**：用于去除相位延迟误差，这类误差通常在带通上表现为近似线性的相位斜坡。延迟校准常在带通校准之前进行，因为它通常可以对整段数据只拟合一次，而实际使用的校准源往往与带通校准相同。<br><br>
</p>

在实际观测中，绝对流量、延迟和带通校准往往可以共用同一个校准源。增益校准理论上也可以使用同一个源，但前提是该源离目标场足够近；而这种情况通常并不成立，因此复增益往往还需要额外的近场校准源。

下图给出了一个典型观测流程的示意图。它并不追求精确，只是帮助理解校准源观测和 1GC 的基本思想。

<img src='figures/observe.png' width=100%>

**图 8.2.1**：一个典型观测中校准源与目标场时间分配的示意图。<a id='cal:fig:time'></a> <!--\label{cal:fig:time}-->

上面的讨论还缺少一个关键问题：我们究竟如何利用这些校准源观测来做实际校准？答案其实相对直接。正如[$\S$ 8.1 &#10142;](../8_Calibration/8_1_calibration_least_squares_problem.ipynb#cal:sec:cal_ls)所示，校准可以被写成一个最小二乘问题，1GC 也不例外。

一个有利条件是：校准源模型通常只包含一个源。若这是一个位于视场中心的点源，那么模型就会极大简化，基本上就是校准源振幅乘上天线增益。对于更复杂的校准源，这种简化不再成立；但只要我们拥有足够准确的源模型，复杂校准源同样可以使用。

当校准源解求出之后，最后一个关键步骤就是把这些解转移到目标场。最简单的做法，是直接把某次校准源观测得到的解应用到紧接着的目标场观测上。更复杂的转移方式，则可以通过对这些解进行插值或随时间拟合曲线来实现。

## 8.2.2 闭合量<a id='cal:sec:closure'></a>

在最小二乘求解器广泛使用之前，还存在其他形式的校准方法。其中一类经典方法依赖于闭合量（closure quantities），最早可追溯到 [<cite data-cite='Jennison1958'>A phase sensitive interferometer technique for the measurement of the Fourier transforms of spatial brightness distributions of small angular extent</cite> &#10548;](http://mnras.oxfordjournals.org/content/118/3/276.short)。虽然今天这类方法已不再是主流，但它引入的量本身仍然非常重要。最常见的闭合量就是闭合相位和闭合振幅。

<img src='figures/triangle.png' width=60%>

**图 8.2.2**：一个用于说明闭合相位的三角形基线结构。<a id='cal:fig:triangle'></a> <!--\label{cal:fig:triangle}-->

为了定义闭合相位，先考虑一个三天线阵列。我们可以把三面天线看作一个三角形的三个顶点，而三角形的边则对应各条基线。每条基线测得的可见度可以写成：

$$\tilde{v}_{pq} = g_pv_{pq}g_{q}^{*}.$$


在上式中，$\tilde{v}_{pq}$ 是测得的可见度，$v_{pq}$ 是真实可见度，$g_p$ 是天线 $p$ 的增益，$g_q^*$ 是天线 $q$ 增益的复共轭。这里的 $p$ 和 $q$ 只是天线编号。把这些量进一步写成振幅 $A$ 和相位 $\phi$ 的形式，就得到：

\begin{eqnarray}
\tilde{v}_{pq} &=& A_{p}e^{-\imath\phi_p}A_{pq}e^{-\imath\phi_{pq}}A_{q}e^{\imath\phi_q} \\
&=& A_{p}A_{pq}A_{q}e^{\imath(-\phi_p-\phi_{pq}+\phi_q)}
\end{eqnarray} 



这些表达式已经足够导出闭合相位和闭合振幅。对于上面的三天线情形，我们分别把三条基线记作 $ij$、$jk$ 和 $ki$。这等价于沿着三角形从顶点 $i$ 顺时针走一圈。对于闭合相位，我们只关心上式的相位部分。

\begin{eqnarray}
\tilde{\phi}_{ij} &=& arg(\tilde{v}_{ij}) = -\phi_i-\phi_{ij}+\phi_j \\
\tilde{\phi}_{jk} &=& arg(\tilde{v}_{jk}) = -\phi_j-\phi_{jk}+\phi_k \\
\tilde{\phi}_{ki} &=& arg(\tilde{v}_{ki}) = -\phi_k-\phi_{ki}+\phi_i
\end{eqnarray} 

这里的 $\tilde{\phi}$ 表示对应基线的总相位。把这三条基线的相位相加，就得到：

<p class=conclusion>
  <font size=4> <b>Closure phase</b></font>
  <br>
  <br>
  \begin{equation}
  \tilde{\phi}_{ij} + \tilde{\phi}_{jk} + \tilde{\phi}_{ki} = -\phi_{ij} - \phi_{jk} - \phi_{ki}.
  \end{equation}
</p>

这一关系就称为*闭合相位*。它有趣之处在于，增益引入的相位项会完全相消，最后只剩下真实相位的和。我们当然也可以对真实相位的指标顺序进行整体或局部交换，只要同时改变符号即可；这相当于对原始的 $\tilde{v}_{pq}$ 取复共轭。

闭合相位至今仍然是非常有用的观测量，因为它能够作为干涉仪性能的诊断工具。对于位于视场中心的点源，理论上每条基线的真实相位都应接近零，因此闭合相位也应接近零。当然在噪声存在时，这一点不可能完全满足；但若某个闭合相位显著偏大，就往往说明系统中存在问题。对阵列中每三个天线组成的所有三角形都计算闭合相位，还可以帮助定位故障天线。例如，如果所有包含天线 $i$ 的闭合相位都明显偏离零，那么很可能就是这面天线需要被标记或排查。

闭合振幅也是一个很有用的观测量。它的推导相对简单，只要直接代入相应表达式即可。对四面天线（这是定义闭合振幅所需的最少数量）$i$、$j$、$k$ 和 $l$，我们可以写出：

$$\frac{\lvert \tilde{v}_{ij} \rvert \lvert \tilde{v}_{kl} \rvert}{\lvert \tilde{v}_{ik} \rvert \lvert \tilde{v}_{jl} \rvert}.$$

如果把类似下面这种形式的表达式代入：

$$ \tilde{v}_{pq} = A_{p}e^{-\imath\phi_p}A_{pq}e^{-\imath\phi_{pq}}A_{q}e^{\imath\phi_q}, $$ 

就会得到下式。注意，在取绝对值时，指数项会自动消失：

$$\frac{\lvert A_iA_{ij}A_{j} \rvert \lvert A_kA_{kl}A_{l} \rvert}{\lvert A_iA_{ik}A_{k} \rvert \lvert A_jA_{jl}A_{l} \rvert}.$$

这样一来，属于增益的振幅项也会被完全约去，我们最终得到闭合振幅关系。

<p class=conclusion>
  <font size=4> <b>Closure amplitude</b></font>
  <br>
  <br>
  \begin{equation}
  \frac{\lvert \tilde{v}_{ij} \rvert \lvert \tilde{v}_{kl} \rvert}{\lvert \tilde{v}_{ik} \rvert \lvert \tilde{v}_{jl} \rvert} = \frac{\lvert A_{ij} \rvert \lvert A_{kl} \rvert}{\lvert A_{ik} \rvert \lvert A_{jl} \rvert}
  \end{equation}
</p>

和闭合相位一样，闭合振幅同样能够消去增益项的贡献，从而给出真实振幅的某种约束。对于位于视场中心的点源，我们预期每条基线测到的振幅应当相同，因此上面的表达式应接近 1。因而，闭合振幅同样可以作为系统状态的诊断量。

***

下一节： [8.3 第二代校准（2GC）](8_3_2gc.ipynb)